---
### **Portfolio formation**

* 수익률 극단치 조정: 상·하위 1% 윈저라이징
* 리밸런싱 주기: 월간 (2000-01-01 \~ 2025-08-31)
* 거래비용 반영:

  * Amihud 유동성 지표 상위 20% 종목: 80bp
  * 기타 종목: 30bp
* 유동성 필터링: 최근 60영업일 거래대금 하위 10% 종목 제외

---
##### **데이터 로드**

In [5]:
import pandas as pd
import numpy as np

# 포트폴리오 수익률 계산을 위한 총수익률 데이터
total_adj_close  = pd.read_csv("./input/total adj close.csv", index_col=0, parse_dates=True)   # 현금배당 반영 수정주가(총수익률 계산)

# 포트폴리오 종목 선정을 위한 Investment factor 데이터
ivol_df             = pd.read_csv("./input/ivol.csv", index_col=0, parse_dates=True)              # Investment factor : ivol

# 시가총액가중 배분비중 계산 데이터
mkt_cap          = pd.read_csv("./input/mkt cap.csv", index_col=0, parse_dates=True)           # 시가총액

# 거래대금 하위 10% 필터링 계산 데이터
trading_value_60 = pd.read_csv("./input/trading value 60.csv", index_col=0, parse_dates=True)  # 60영업일 평균 거래대금

# Amihud illiquidity 계산 데이터
daily_ret        = pd.read_csv("./input/daily ret.csv", index_col=0, parse_dates=True)         # 일간 수익률 월별 데이터
trading_value    = pd.read_csv("./input/trading value.csv", index_col=0, parse_dates=True)     # 월말 거래대금

---
##### **총수익률 데이터 전처리**

In [6]:
# 현금배당반영 수정주가 데이터를 이용해 월간 총수익률 계산
ret_m = total_adj_close.ffill().pct_change()

# 수익률 윈저라이징
ret_wins = ret_m.clip(
    lower=ret_m.quantile(0.01),
    upper=ret_m.quantile(0.99),
    axis=1
)

In [7]:
ret_wins.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,KB금융,기아,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-05-31,0.012616,0.154147,-0.118644,-0.019011,0.017566,-0.015082,0.008708,0.167790,-0.011062,0.393782,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2025-06-30,0.070476,0.429383,0.038462,-0.038760,0.045623,0.108931,0.060904,0.070877,0.083893,0.695167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005097,0.0
2025-07-31,0.193980,-0.063356,0.266424,0.075605,0.174528,0.046683,0.144691,0.000000,0.055728,-0.040936,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2025-08-31,-0.023810,-0.016453,-0.079739,-0.061856,-0.112450,0.032864,0.060143,-0.016435,0.034213,-0.059451,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2025-09-09,0.025825,0.070632,-0.014205,0.039960,0.069005,-0.004545,-0.030769,0.015712,-0.002836,0.008104,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


---
##### **리밸런싱 날짜 설정**

In [8]:
# 백테스트 기간을 설정하여 월말 날짜 리스트 생성
start_point = '1999-12-31'
end_point   = '2025-08-31'

month_ends = ivol_df.resample('ME').last().index
month_ends = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

month_ends

DatetimeIndex(['1999-12-31', '2000-01-31', '2000-02-29', '2000-03-31',
               '2000-04-30', '2000-05-31', '2000-06-30', '2000-07-31',
               '2000-08-31', '2000-09-30',
               ...
               '2024-11-30', '2024-12-31', '2025-01-31', '2025-02-28',
               '2025-03-31', '2025-04-30', '2025-05-31', '2025-06-30',
               '2025-07-31', '2025-08-31'],
              dtype='datetime64[ns]', name='Date', length=309, freq='ME')

---
##### **포트폴리오 수익률 계산 - 1st Cycle**

Investment factor 설정

In [9]:
factor_df = ivol

In [10]:
factor_df

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
1991-08-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-30,0.046126,0.077278,NaN,0.067957,0.125740,0.084412,NaN,0.077852,0.071672,0.189411,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.116061,NaN
2025-05-31,0.046472,0.077139,NaN,0.067908,0.123177,0.086175,NaN,0.077817,0.071851,0.191410,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.115267,NaN
2025-06-30,0.047999,0.081132,NaN,0.065427,0.123540,0.086487,NaN,0.078068,0.072243,0.199805,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.114454,NaN


포트폴리오 설정

In [11]:
n                = 20     # 종목 수
high_cost        = 0.008  # 80bp
low_cost         = 0.003  # 30bp

initial_NAV      = 1   # 초기값
NAV              = initial_NAV

portfolio_return = pd.Series(dtype=float)
total_trade      = pd.Series(dtype=float)

첫 Cycle 날짜 설정

In [12]:
i = 0                               # 날짜 인덱스 설정
start_date = month_ends[i]          # 리밸런싱 기준일 (전월말)
end_date = month_ends[i + 1]        # 다음 리밸런싱 시점 (다음 월말)

거래대금 하위 10% 필터링

In [13]:
series = trading_value_60.loc[start_date]
threshold = series.quantile(0.1)   # 하위 10% threshold
filtered = series[series > threshold]

# filtered의 index만 남기기
factor_filtered = factor_df.loc[start_date, filtered.index]

포트폴리오 종목 선정

In [14]:
basket = factor_filtered.nsmallest(n)      # 변동성 하위 20개 종목

Amihud illiquidity 계산

In [15]:
illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])

# 상위 20% (illiquid)
threshold = illiq.quantile(0.8)
illiquid_top20 = illiq[illiq >= threshold].index

거래비용 반영

In [16]:
# 목표 자산배분 비중 계산

# 시총가중
# cap_seg        = mkt_cap.loc[start_date, basket.index]
# target_weights = (cap_seg / cap_seg.sum())

# 동일가중
target_weights = pd.Series(1/len(basket), index=basket.index, name=start_date)

In [17]:
target_weights

KG스틸        0.05
한국전력        0.05
유한양행        0.05
제일파마홀딩스     0.05
신영와코루       0.05
POSCO홀딩스    0.05
메리츠종금       0.05
녹십자홀딩스      0.05
동양화학        0.05
대성합동지주      0.05
현대약품        0.05
DN오토모티브     0.05
SK.1        0.05
부국증권        0.05
한솔홀딩스       0.05
삼영전자        0.05
CJ          0.05
코오롱유화       0.05
LS          0.05
한솔케미칼       0.05
Name: 1999-12-31 00:00:00, dtype: float64

In [18]:
# 첫 거래 가정
prev_weights   = pd.Series(0, index=basket.index, name=start_date)

In [19]:
prev_weights

KG스틸        0
한국전력        0
유한양행        0
제일파마홀딩스     0
신영와코루       0
POSCO홀딩스    0
메리츠종금       0
녹십자홀딩스      0
동양화학        0
대성합동지주      0
현대약품        0
DN오토모티브     0
SK.1        0
부국증권        0
한솔홀딩스       0
삼영전자        0
CJ          0
코오롱유화       0
LS          0
한솔케미칼       0
Name: 1999-12-31 00:00:00, dtype: int64

In [20]:
# 1. 비중 차이
all_index = target_weights.index.union(prev_weights.index)   # 전체 종목 universe = 두 시점 종목의 합집합

target_w = target_weights.reindex(all_index, fill_value=0)   # 비중 시리즈를 동일한 index로 확장
prev_w   = prev_weights.reindex(all_index, fill_value=0)     # 없으면 0으로 채움

delta_w = target_w - prev_w                                  # 차이 계산

In [21]:
delta_w

KG스틸        0.05
한국전력        0.05
유한양행        0.05
제일파마홀딩스     0.05
신영와코루       0.05
POSCO홀딩스    0.05
메리츠종금       0.05
녹십자홀딩스      0.05
동양화학        0.05
대성합동지주      0.05
현대약품        0.05
DN오토모티브     0.05
SK.1        0.05
부국증권        0.05
한솔홀딩스       0.05
삼영전자        0.05
CJ          0.05
코오롱유화       0.05
LS          0.05
한솔케미칼       0.05
Name: 1999-12-31 00:00:00, dtype: float64

In [22]:
# 2. 거래금액 (매수/매도 절댓값)
trade_amounts = abs(delta_w) * NAV

# 3. 거래비용률 결정 (illiquid 종목은 높은 비용 적용)
cost_rate = np.where(delta_w.index.isin(illiquid_top20), high_cost, low_cost)  # 80bp vs 30bp

# 4. 총 거래비용
trade_cost = (trade_amounts * cost_rate).sum()

# 5. NAV 업데이트
NAV_new = NAV - trade_cost

# 6. 새 포트폴리오 가치
current_portfolio_value = target_weights * NAV_new

In [23]:
current_portfolio_value

KG스틸        0.049838
한국전력        0.049838
유한양행        0.049838
제일파마홀딩스     0.049838
신영와코루       0.049838
POSCO홀딩스    0.049838
메리츠종금       0.049838
녹십자홀딩스      0.049838
동양화학        0.049838
대성합동지주      0.049838
현대약품        0.049838
DN오토모티브     0.049838
SK.1        0.049838
부국증권        0.049838
한솔홀딩스       0.049838
삼영전자        0.049838
CJ          0.049838
코오롱유화       0.049838
LS          0.049838
한솔케미칼       0.049838
Name: 1999-12-31 00:00:00, dtype: float64

In [24]:
# 당기 포트폴리오 최종 가치 계산
ret_seg              = ret_wins.loc[end_date, basket.index]
next_portfolio_value = current_portfolio_value * (ret_seg + 1)

# 당기 포트폴리오 수익률 계산
NAV_new       = next_portfolio_value.sum()
portfolio_ret = NAV_new / NAV - 1

In [25]:
print('포트폴리오 월초 NAV :', NAV)
print('포트폴리오 월말 NAV :', round(NAV_new, 3))
print('포트폴리오 월 수익률 :', round(portfolio_ret, 3))

포트폴리오 월초 NAV : 1
포트폴리오 월말 NAV : 0.978
포트폴리오 월 수익률 : -0.022


In [27]:
# prev_portfolio 업데이트
prev_portfolio = next_portfolio_value

# NAV 업데이트
NAV = NAV_new

# 총 거래금액 저장
total_trade.loc[start_date] = trade_amounts.sum()

# 포트폴리오 수익률 저장
portfolio_return.loc[end_date] = portfolio_ret

In [28]:
prev_portfolio

KG스틸        0.045684
한국전력        0.049129
유한양행        0.046525
제일파마홀딩스     0.054907
신영와코루       0.049645
POSCO홀딩스    0.053027
메리츠종금       0.057618
녹십자홀딩스      0.054668
동양화학        0.043430
대성합동지주      0.049393
현대약품        0.062105
DN오토모티브     0.039888
SK.1        0.040709
부국증권        0.053066
한솔홀딩스       0.049591
삼영전자        0.044268
CJ          0.038416
코오롱유화       0.048838
LS          0.053692
한솔케미칼       0.043227
dtype: float64

---
##### **전체 기간 포트폴리오 수익률 계산**

In [47]:
for i in range(1, len(month_ends) - 1):

    # 날짜 업데이트
    start_date = month_ends[i]
    end_date = month_ends[i + 1] 

    # 거래대금 하위 10% 필터링
    series = trading_value_60.loc[start_date]
    threshold = series.quantile(0.1)
    filtered = series[series > threshold]
    factor_filtered = factor_df.loc[start_date, filtered.index]

    # 종목 선정
    current_long_basket = factor_filtered.nsmallest(n)

    # Amihud illiquidity 계산
    illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
    threshold = illiq.quantile(0.8)
    illiquid_top20 = illiq[illiq >= threshold].index

    # basket 설정
    basket = current_long_basket

    # 자산배분 비중 계산
    prev_weights   = prev_portfolio / prev_portfolio.sum()
    target_weights = pd.Series(1/len(basket), index=basket.index, name=start_date)

    # 거래비용 반영
    all_index = target_weights.index.union(prev_weights.index)
    target_w = target_weights.reindex(all_index, fill_value=0)
    prev_w   = prev_weights.reindex(all_index, fill_value=0)
    delta_w = target_w - prev_w
    trade_amounts = abs(delta_w) * NAV
    cost_rate = np.where(delta_w.index.isin(illiquid_top20), high_cost, low_cost)
    trade_cost = (trade_amounts * cost_rate).sum()
    NAV_new = NAV - trade_cost
    current_portfolio_value = target_weights * NAV_new

    # 당기 포트폴리오 최종 가치 계산
    ret_seg              = ret_wins.loc[end_date, basket.index]
    next_portfolio_value = current_portfolio_value * (ret_seg + 1)
    
    # 당기 포트폴리오 수익률 계산
    NAV_new       = next_portfolio_value.sum()
    portfolio_ret = NAV_new / NAV - 1

    # prev_portfolio 업데이트
    prev_portfolio = next_portfolio_value

    # NAV 업데이트
    NAV = NAV_new

    # 총 거래금액 저장
    total_trade.loc[start_date] = trade_amounts.sum()
    
    # 포트폴리오 수익률 저장
    portfolio_return.loc[end_date] = portfolio_ret

In [50]:
portfolio_return.tail()

Date
2025-04-30    0.051666
2025-05-31    0.068698
2025-06-30    0.031056
2025-07-31    0.040848
2025-08-31   -0.027382
dtype: float64

---
##### **NAV 계산 및 OUTPUT 데이터 전처리**

In [77]:
portfolio_NAV = (1 + portfolio_return).cumprod() * initial_NAV

In [79]:
# Portfolio 데이터 생성
df = pd.concat(
    [portfolio_return, portfolio_NAV, total_trade], 
    axis=1
)

df.columns = ["Return", "NAV", "Trade"]
df.index.name = "Date"

# 초기값 저장
df.loc[df.index[0], "NAV"] = initial_NAV



In [81]:
df

,Return,NAV,Trade
Date,,,
1999-12-31,NaN,1.000000,1.000000
2000-01-31,-0.022175,0.977825,0.391422
2000-02-29,-0.008684,0.969333,0.535475
2000-03-31,0.063178,1.030573,0.401478
2000-04-30,-0.143816,0.882361,0.140339
...,...,...,...
2025-04-30,0.051666,39.716165,9.318065
2025-05-31,0.068698,42.444596,15.376515
2025-06-30,0.031056,43.762754,10.362672


---
##### **데이터 저장**

In [82]:
# Output 폴더
df.to_csv("./output/portfolio.csv", index=True, encoding="utf-8-sig")

# 데이터가 필요한 단계의 input 폴더
df.to_csv("../Step4 성과 평가 (Performance evalution)/input/portfolio.csv", index=True, encoding="utf-8-sig")